In [ ]:
# 演示 阻塞
import socket

def step1_the_blocking_dead():
    """同步阻塞：线程被OS强制休眠，彻底丧失执行权"""
    # 创建一个真实的TCP直接连本地的不存在空端口
    sock = socket.socket()
    sock.settimeout(120)
    print("[阻塞模型] 尝试连接一个无法响应的目标...")
    try:
        sock.connect(("1.1.1.1", 9999)) # connect()是一个阻塞式系统调用，连接一个不存在的地址
        print(f"执行简单计算：{1+1}") # <--- 这里永远不会执行，因地址不存在，永远等不到成功
    except (ConnectionError, socket.timeout) as e:
        print(f"[阻塞模型] 线程被挂起 {sock.timeout} 秒。状态: {e}")

step1_the_blocking_dead()

[阻塞模型] 尝试连接一个无法响应的目标...
[阻塞模型] 线程被挂起 120.0 秒。状态: [WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。


In [ ]:
# 演示 非阻塞
import socket
def step2_nonblocking_and_busy_wait():
    """非阻塞：O_NONBLOCK 解放线程"""
    sock = socket.socket()
    sock.setblocking(False) # <--- 核心前提1：OS 支持 O_NONBLOCK
    print("[非阻塞模型] 尝试连接...")
    try:
        sock.connect(("1.1.1.1", 9999))
    except BlockingIOError:
        # 连接尚未建立，OS立刻交还控制权，线程没被挂起！
        print("[非阻塞模型] 连接进行中，线程没卡死，可以干别的！")
    print("[非阻塞模型] 但怎么知道连接成功了？只能死循环问 OS...")
    import time
    start = time.perf_counter()
    loops = 0
    
    # 模拟轮询等待（这就是 CPU 空转的元凶）
    while time.perf_counter() - start < 0.01: # 跑 10 毫秒
        try:
            # 不断尝试发送数据探测连接是否完成
            sock.send(b"") 
            break # 没抛异常，说明连接成功了
        except (BlockingIOError, OSError):
            loops += 1 # 没好，继续死循环问
            
    print(f"[非阻塞模型] 10ms 内死循环问了 OS {loops} 次！CPU 满载空转。")

step2_nonblocking_and_busy_wait()

[非阻塞模型] 尝试连接...
[非阻塞模型] 连接进行中，线程没卡死，可以干别的！
执行简单计算：2


In [ ]:
# 演示 多路复用
import socket
import selectors # <--- 核心前提2：OS 提供 select/epoll
import time

def step3_select_causes_eventloop():
    """多路复用解决空转，但逼迫出 EventLoop 和 回调"""
    sel = selectors.DefaultSelector()
    
    # 准备两个非阻塞 socket 模拟并发 I/O
    sock1 = socket.socket(); sock1.setblocking(False)
    sock2 = socket.socket(); sock2.setblocking(False)
    
    try: sock1.connect(("1.1.1.1", 80))
    except BlockingIOError: pass
    try: sock2.connect(("8.8.8.8", 80))
    except BlockingIOError: pass

    # --- I4 影响：就绪事件需绑定完成动作 (回调契约) ---
    def on_sock1_ready(sock, mask):
        try: sock.send(b""); print("  [回调1] sock1 连接成功！处理 sock1 的业务。")
        except: pass
        sel.unregister(sock) # 处理完就取消注册

    def on_sock2_ready(sock, mask):
        try: sock.send(b""); print("  [回调2] sock2 连接成功！处理 sock2 的业务。")
        except: pass
        sel.unregister(sock)

    # 注册：告诉 OS 监听这两个 fd，并绑好对应的回调契约
    sel.register(sock1, selectors.EVENT_WRITE, on_sock1_ready)
    sel.register(sock2, selectors.EVENT_WRITE, on_sock2_ready)

    # --- I3 影响：控制权真空，需要调度中枢 (EventLoop) ---
    print("[调度中枢] 接管控制权，进入事件循环...")
    
    # 这是一个极简的 EventLoop 结构
    start = time.perf_counter()
    while time.perf_counter() - start < 0.05: # 模拟运行 50ms
        # 1. 问 OS：谁就绪了？(timeout=0 立即返回，不阻塞线程)
        events = sel.select(timeout=0) 
        
        if not events:
            # 没事件，线程可以去做别的事（不会死循环吃 CPU，因为可以算时间休眠）
            continue
            
        # 2. 调度中枢分发：接下来执行谁？
        for key, mask in events:
            callback = key.data   # 取出绑定的契约
            callback(key.fileobj, mask) # 3. 履行契约

    print("[调度中枢] 循环结束。CPU 没空转，线程没挂起。")
    
    sel.close(); sock1.close(); sock2.close()

step3_select_causes_eventloop()

### 这组代码的逻辑闭环

1.  **演进 1**：证明阻塞=线程被挂，CPU 闲置。
2.  **演进 2**：`O_NONBLOCK` 解决挂起，但带来轮询空转，CPU 满载干没用的事。
3.  **演进 3**：`select/epoll` 让 OS 代为监听，有事件才返回。**线程既不挂起，也不空转。**
4.  **点睛之笔**：在演进 3 中，`sel.select()` 返回后，程序面临一堆就绪事件，必须通过 `key.data` 取出预先绑定的 `callback` 并执行。这就**用代码实锤了你推导的逻辑**——必须有一个 `while` 循环作为**调度中枢**，必须把逻辑切分成 `on_sock_ready` 作为**完成契约**。

没有任何多线程干扰，全是 OS 原生机制的步步紧逼。